In [ ]:
# NOTEBOOK 35 - step 1b-ii: which lag and horizon carry the regional signal, and how much does it add to a simple Delhi forecast?  (no neural model)
# Regional signal = the mean log PM2.5 of the regional stations that reported in that hour (at least 3 of 8), taken L hours earlier,
# minus Delhi's log PM2.5 now.  Target = Delhi city-mean change over the next h hours.  Winter months only (Oct-Mar).
#   Part 1: partial correlation (after removing Delhi's own level and its recent change) for lags 0..36 h and horizons 3..24 h.
#   Part 2: out-of-sample value.  A plain linear model of the Delhi change is trained on winter hours BEFORE 8 Dec 2025 and scored on
#           winter hours AFTER it (the Delhi test period), with and without the regional features. The gain in R2 is what the signal is worth.
# Controls (in both parts): Delhi's own level, its 6 h and 24 h change, HOUR OF DAY (23 dummies) and month - so a regional feature only gets credit
# for information beyond Delhi's own history and the calendar (the first version lacked the hour-of-day control and was inflated by the daily cycle).
# Needs regional_cache/regional_dataset.npz (notebook 33). Reads only; writes nothing.
# HOW TO CHECK: this cell only loads the data.
import os, numpy as np, pandas as pd
V2_DIR = r'D:\FYP\Eco-Twin\v2_clean'
z = np.load(os.path.join(V2_DIR, 'regional_cache', 'regional_dataset.npz'))
PM = z['pm']; GRID = pd.DatetimeIndex(z['grid']); T, R = PM.shape
ds = pd.read_csv(os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv'), parse_dates=['Timestamp'])
DEL = np.log1p(ds.pivot_table(index='Timestamp', columns='Station', values='pm2.5').reindex(GRID).mean(axis=1).values)
L_PM = np.log1p(PM)
cnt = np.isfinite(L_PM).sum(axis=1)
with np.errstate(all='ignore'):
    RM = np.where(cnt >= 3, np.nanmean(L_PM, axis=1), np.nan)               # regional mean log PM2.5, only when at least 3 stations reported
winter = np.isin(GRID.month, [10, 11, 12, 1, 2, 3]); day = np.array((GRID - GRID[0]).days)
SPLIT = int(np.searchsorted(GRID, pd.Timestamp('2025-12-08')))
print('regional mean available in', int(np.isfinite(RM).sum()), 'of', T, 'hours | winter hours', int(winter.sum()), '| split index (8 Dec 2025):', SPLIT)

In [ ]:
LAGS = [0, 1, 2, 3, 6, 9, 12, 18, 24, 36]; HORIZONS = [3, 6, 12, 18, 24]
def shift(a, k):                                                              # value k hours earlier (NaN where it does not exist)
    out = np.full_like(a, np.nan); out[k:] = a[:T - k] if k else a; return out
def rows_for(h, lags):
    t = np.arange(36, T - h)
    tgt = DEL[t + h] - DEL[t]
    hr = np.eye(24)[GRID.hour.values[t]][:, 1:]                                          # hour of day (23 dummies): removes the daily cycle
    mo = np.column_stack([np.sin(2 * np.pi * (GRID.month.values[t] - 1) / 12), np.cos(2 * np.pi * (GRID.month.values[t] - 1) / 12)])
    ctrl = np.column_stack([DEL[t], DEL[t] - DEL[t - 6], DEL[t] - DEL[t - 24], hr, mo])   # Delhi's own level and recent changes + hour of day + month
    feats = np.column_stack([shift(RM, L)[t] - DEL[t] for L in lags])
    ok = np.isfinite(tgt) & np.isfinite(ctrl).all(1) & np.isfinite(feats).all(1) & winter[t]
    return t[ok], tgt[ok], ctrl[ok], feats[ok]
def partial_r(tgt, ctrl, f):
    X = np.column_stack([np.ones(len(tgt)), ctrl]); res = lambda y: y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
    return float(np.corrcoef(res(tgt), res(f))[0, 1])

print('PART 1 - partial correlation of the regional signal at lag L (rows) with the Delhi change over the next h hours (columns)')
print(f"{'lag L (h)':>10}" + ''.join(f'{"next " + str(h) + " h":>11}' for h in HORIZONS) + '      hours used (h=12)')
tab = np.zeros((len(LAGS), len(HORIZONS))); n12 = None
for j, h in enumerate(HORIZONS):
    for i, L in enumerate(LAGS):
        t, tgt, ctrl, f = rows_for(h, [L]); tab[i, j] = partial_r(tgt, ctrl, f[:, 0])
        if h == 12 and L == 0: n12 = len(t)
for i, L in enumerate(LAGS): print(f'{L:>10}' + ''.join(f'{tab[i, j]:>+11.3f}' for j in range(len(HORIZONS))) + (f'      {n12}' if i == 0 else ''))
print('best lag per horizon:', {f'{h} h': LAGS[int(np.argmax(tab[:, j]))] for j, h in enumerate(HORIZONS)})

# HOW TO READ: a ridge of high values at lag 6-12 h for the 12-24 h horizons means the regional readings lead Delhi by half a day and are worth using.

In [ ]:
# PART 2 - what is it worth out of sample?  Linear model of Delhi's change; train = winter before 8 Dec 2025, test = winter after
USE = [0, 6, 12, 24]
def r2(y, p, y0): return 1 - np.sum((y - p) ** 2) / np.sum((y - y0.mean()) ** 2)
print(f"{'horizon':>8}{'train rows':>12}{'test rows':>11}{'R2 Delhi only':>15}{'R2 + regional':>15}{'gain':>8}{'RMSE (log) only':>17}{'RMSE (log) + regional':>23}")
out = {}
for h in HORIZONS:
    t, tgt, ctrl, f = rows_for(h, USE)
    tr, te = t < SPLIT - h, t >= SPLIT                                          # keep training targets strictly before the split
    Xb = lambda c, ff=None: np.column_stack([np.ones(len(c)), c] + ([ff] if ff is not None else []))
    wb = np.linalg.lstsq(Xb(ctrl[tr]), tgt[tr], rcond=None)[0]; wr = np.linalg.lstsq(Xb(ctrl[tr], f[tr]), tgt[tr], rcond=None)[0]
    pb, pr = Xb(ctrl[te]) @ wb, Xb(ctrl[te], f[te]) @ wr
    a, b = r2(tgt[te], pb, tgt[tr]), r2(tgt[te], pr, tgt[tr])
    ra, rb = float(np.sqrt(np.mean((tgt[te] - pb) ** 2))), float(np.sqrt(np.mean((tgt[te] - pr) ** 2)))
    out[h] = (a, b); print(f'{h:>6} h{tr.sum():>12}{te.sum():>11}{a:>15.3f}{b:>15.3f}{b - a:>+8.3f}{ra:>17.3f}{rb:>23.3f}')
print()
print('lags used in the regional features:', USE, 'hours. R2 is on the held-out winter (8 Dec 2025 onward); a positive gain is real out-of-sample value.')

# HOW TO READ: a gain of about 0.02-0.05 R2 on the 12-24 h horizons would be a useful addition for a neural model; under 0.01 means the
# regional stations add little beyond what Delhi's own recent history already says.